# SMIRK on Kaggle: setup and demo

Installs SMIRK (code from github.com/THUONGVIP09/smirk, branch occlusion-aware), downloads FLAME 2020 with your FLAME
account and the pretrained SMIRK_em1.pt, then reconstructs some test faces of clean-faces.

Before running:
- Settings: **Accelerator GPU T4 x1**, **Internet on**.
- *Add-ons → Secrets*: add `FLAME_USERNAME` (FLAME account email) and `FLAME_PASSWORD`, and enable them for this notebook.
- Inputs: `clean-faces`, `affectnet-crops`, `expw-crops`. If a dataset with a `pytorch3d-*.whl` (output of a previous
  run of this notebook) is added as input, PyTorch3D is installed from it instead of being built (saves ~20-30 min).

FLAME and SMIRK may not be redistributed: keep this notebook and its outputs **Private**.

## 1. Environment

In [ ]:
import sys, os, glob, subprocess
import torch, numpy
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda,
      '| numpy', numpy.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'enable the GPU accelerator'

## 2. Get the SMIRK code

In [ ]:
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1

## 3. Python packages

timm is pinned to the version SMIRK was trained with (the encoder weights are stored with its layer names).
PyTorch3D has no wheel for this Python / PyTorch, so it is built from source once (~20-30 min) and the wheel is kept in
the output, to be reused as an input dataset next time.

In [ ]:
!pip install -q timm==0.9.16 iopath gdown

wheels = glob.glob('/kaggle/input/**/pytorch3d-*.whl', recursive=True)
if wheels:
    print('installing', wheels[0])
    !pip install -q {wheels[0]}
else:
    os.environ.update(FORCE_CUDA='1', TORCH_CUDA_ARCH_LIST='7.5', MAX_JOBS='4')  # T4 = compute capability 7.5
    !pip wheel --no-build-isolation --no-deps -w /kaggle/working/wheels "git+https://github.com/facebookresearch/pytorch3d.git@main"
    !pip install -q /kaggle/working/wheels/pytorch3d-*.whl

import pytorch3d
print('pytorch3d', pytorch3d.__version__)

## 4. FLAME 2020 (downloaded with the account stored in Kaggle Secrets)

In [ ]:
from kaggle_secrets import UserSecretsClient
from urllib.parse import quote

secrets = UserSecretsClient()
post_data = 'username=' + quote(secrets.get_secret('FLAME_USERNAME')) + '&password=' + quote(secrets.get_secret('FLAME_PASSWORD'))
subprocess.run(['wget', '-q', '--post-data', post_data, '--no-check-certificate', '-O', '/tmp/FLAME2020.zip',
                'https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2020.zip&resume=1'], check=True)
del post_data

!unzip -q -o /tmp/FLAME2020.zip -d /tmp/smirk/assets/FLAME2020/
!ls -la /tmp/smirk/assets/FLAME2020/
assert os.path.exists('/tmp/smirk/assets/FLAME2020/generic_model.pkl'), 'FLAME download failed: check the FLAME_USERNAME / FLAME_PASSWORD secrets'

## 5. Pretrained SMIRK (SMIRK_em1.pt, from the authors' Google Drive)

In [ ]:
!mkdir -p /tmp/smirk/pretrained_models
!gdown -q 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O /tmp/smirk/pretrained_models/SMIRK_em1.pt
!ls -la /tmp/smirk/pretrained_models/

## 6. Load FLAME, renderer and encoder

In [ ]:
os.chdir('/tmp/smirk')
sys.path.insert(0, '/tmp/smirk')
from src.smirk_encoder import SmirkEncoder
from src.FLAME.FLAME import FLAME
from src.renderer.renderer import Renderer

DEVICE = 'cuda'
flame = FLAME().to(DEVICE)
print('FLAME shapedirs', tuple(flame.shapedirs.shape), '(expected (5023, 3, 350))')
renderer = Renderer().to(DEVICE)

encoder = SmirkEncoder().to(DEVICE).eval()
checkpoint = torch.load('pretrained_models/SMIRK_em1.pt', map_location=DEVICE)
encoder.load_state_dict({k.replace('smirk_encoder.', ''): v for k, v in checkpoint.items() if k.startswith('smirk_encoder')})
print('SMIRK encoder loaded')

## 7. Reconstruct some clean test faces (2 per expression)

In [ ]:
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt

clean_dir = os.path.dirname(glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True)[0])
meta = pd.read_csv(f'{clean_dir}/clean_meta.csv', dtype={'name': str})
landmark_data = np.load(f'{clean_dir}/landmarks.npz')
landmarks_of = dict(zip(landmark_data['keys'], landmark_data['landmarks']))

image_dirs = {}
for meta_csv in glob.glob('/kaggle/input/**/meta.csv', recursive=True):
    folder = os.path.dirname(meta_csv)
    name = {'expw_crops': 'expw', 'affectnet_crops': 'affectnet', 'rafdb_crops': 'rafdb'}.get(os.path.basename(folder))
    if name:
        image_dirs[name] = os.path.join(folder, 'images')
print('image folders:', image_dirs)

def smirk_crop(image, landmarks, scale=1.4, size=224):
    # same crop as demo.py / BaseDataset.crop_face
    left, top = landmarks.min(axis=0); right, bottom = landmarks.max(axis=0)
    old_size = (right - left + bottom - top) / 2
    center = np.array([(left + right) / 2.0, (top + bottom) / 2.0])
    s = int(old_size * scale)
    src = np.float32([[center[0] - s / 2, center[1] - s / 2], [center[0] - s / 2, center[1] + s / 2], [center[0] + s / 2, center[1] - s / 2]])
    dst = np.float32([[0, 0], [0, size - 1], [size - 1, 0]])
    return cv2.warpAffine(image, cv2.getAffineTransform(src, dst), (size, size), borderValue=0)

test = meta[(meta['split'] == 'test') & meta['dataset'].isin(list(image_dirs))]
sample = test.groupby('expression', group_keys=False).apply(lambda g: g.sample(min(2, len(g)), random_state=0))
crops = [smirk_crop(cv2.imread(f"{image_dirs[r['dataset']]}/{r['name']}.png")[..., ::-1],
                    landmarks_of[f"{r['dataset']}/{r['name']}"][:, :2]) for _, r in sample.iterrows()]

images = torch.tensor(np.stack(crops)).permute(0, 3, 1, 2).float().div(255).to(DEVICE)
with torch.no_grad():
    outputs = encoder(images)
    rendered = renderer.forward(flame.forward(outputs)['vertices'], outputs['cam'])['rendered_img']

fig, axes = plt.subplots(3, len(crops), figsize=(2 * len(crops), 6.5))
for i, (_, r) in enumerate(sample.iterrows()):
    img, ren = crops[i] / 255.0, rendered[i].permute(1, 2, 0).cpu().numpy()
    for ax, picture in zip(axes[:, i], [img, ren, np.where(ren.sum(-1, keepdims=True) > 0, 0.6 * ren + 0.4 * img, img)]):
        ax.imshow(np.clip(picture, 0, 1)); ax.axis('off')
    axes[0, i].set_title(f"{r['dataset']}\n{r['expression']}", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Save

*Save Version → Save & Run All*. The output keeps `wheels/pytorch3d-*.whl`: create a **Private** dataset from it
(for example `pytorch3d-wheel`) and add it as input next time to skip the PyTorch3D build.